In [32]:
import pandas as pd, altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# Values read from the published Think Labour chart (±0.3pp); national = remainder
df = pd.DataFrame({
    "country": ["United Kingdom", "Italy", "Spain", "Japan", "United States", "France", "Germany", "Canada"],
    "sub":     [1.2, 2.5, 7.0, 9.2, 9.2, 7.6, 24.4, 39.8],
    "other":   [0.0, 1.1, 2.4, 4.4, 7.4, 14.8, 21.0, 9.7],
})
df = df[df["country"] != "Spain"]   # G7 only
df["cum"] = df["sub"] + df["other"]
df["total"] = 100
df["tier"] = "Sub-national government"   # single category -> first theme colour (teal)
order = df.sort_values("cum")["country"].tolist()

LIGHT, MID, TEXT = "#CBD5E1", "#94A3B8", "#676A86"   # neutral greys (only hard-coded colours)

x = alt.X("country:N", sort=order, title=None, axis=alt.Axis(labelAngle=0))

national = (alt.Chart(df).mark_bar(color=LIGHT, stroke="white", strokeWidth=1)
            .encode(x=x, y=alt.Y("total:Q", scale=alt.Scale(domain=[0, 100]), title=None,
                                 axis=alt.Axis(labelExpr="datum.value + '%'"))))
other = (alt.Chart(df).mark_bar(color=MID, stroke="white", strokeWidth=1)
         .encode(x=x, y="cum:Q"))
sub = (alt.Chart(df).mark_bar(stroke="white", strokeWidth=1)
       .encode(x=x, y="sub:Q", color=alt.Color("tier:N", legend=None)))

main = (national + other + sub).properties(width=400, height=300)

# legend key: teal swatch uses the same single-category theme colour as the bars
kx = alt.X("x:Q", scale=alt.Scale(domain=[0, 400]), axis=None)
ky = alt.Y("y:Q", scale=alt.Scale(domain=[0, 20]), axis=None)
key_df = pd.DataFrame({"x": [4, 152, 284], "y": [10, 10, 10],
                       "label": ["Sub-national government", "Other state agencies", "National government"]})

swatch_sub = (alt.Chart(key_df.iloc[[0]].assign(tier="Sub-national government"))
              .mark_square(size=90, opacity=1)
              .encode(x=kx, y=ky, color=alt.Color("tier:N", legend=None)))
swatch_other = (alt.Chart(key_df.iloc[[1]]).mark_square(size=90, opacity=1, color=MID)
                .encode(x=kx, y=ky))
swatch_nat = (alt.Chart(key_df.iloc[[2]]).mark_square(size=90, opacity=1, color=LIGHT)
              .encode(x=kx, y=ky))
key_text = (alt.Chart(key_df).mark_text(align="left", baseline="middle", dx=9, fontSize=10, color=TEXT)
            .encode(x=kx, y=ky, text="label:N"))

key = (swatch_sub + swatch_other + swatch_nat + key_text).properties(width=400, height=20)

chart = alt.vconcat(key, main, spacing=6).properties(
    padding={"left": 5, "right": 5, "top": 5, "bottom": 5},
)
chart

styles.save(chart, path=".", name="g7_debt_by_tier", svg=True)